In [1]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.io import MemoryFile
import rioxarray as rxr
import s3fs
import fsspec
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import gc
import psutil
from tqdm import tqdm
import re
import sys
from pathlib import Path
import glob

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")
print(f"Rasterio version: {rasterio.__version__}")

✅ Libraries imported successfully!
Boto3 version: 1.39.11
Rasterio version: 1.4.3


In [2]:
# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog,
    export_COG_PROFILE
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing_local import (
    process_file_batch,
    print_batch_summary
)

from memory_utils import (
    get_memory_usage,
    calculate_optimal_chunk_size,
    estimate_chunk_memory,
    format_bytes

)

from convert_utilities_local import (
    convert_to_proper_CRS_and_cogify_chunked
)
    
print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Memory monitoring utilities loaded
✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


In [3]:
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'

###

EVENT_NAME = "202502_Flood_OhioValley"
local_path = "/home/jovyan/files_to_manually_convert/"

In [4]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = export_COG_PROFILE()


# Chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}

In [5]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

In [6]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name=BUCKET, verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name=BUCKET, verbose=True)

✅ S3 client initialized successfully
   Found 68 accessible buckets
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files


True

In [7]:
def make_regex_dict(keys, regexes, products):
    ret = {}
    for i in range(len(products)):
        matches = []
        for key in keys:
            filename = key.split("/")[-1]
            match = re.search(regexes[i], filename)
            if match is not None:
                matches.append(key)
        if matches != []:
            ret[products[i]] = matches
    return ret

In [12]:
def create_cog_filename(filename, event):
    sname = filename.split("/")[-1].replace(".tif", "").split("_")
    date = datetime.strptime(sname[2], "%Y%m%d")
    new_dt_format = date.strftime("%Y-%m-%d_day")
    cog_filename = f"{event}_{sname[0]}_{sname[1]}_{sname[3]}_{new_dt_format}.tif"
    return cog_filename

In [13]:
def simple_process_files(file_list, rename_func, target_dir, event):
    """
    Simple wrapper to process files with minimal code.
    
    Args:
        keys: List of all S3 keys
        filter_str: Can be:
            - String to filter files (e.g. 'S1_WTR')
            - Regex pattern object (e.g. re.compile(r'.*S2A.*mosaic'))
            - Callable function that returns True/False
        rename_func: Your custom rename function
        target_dir: Target directory (e.g. "Sentinel-1/opera_dswx")
        EVENT_NAME: Event name
    
    Returns:
        Processing results DataFrame
    """
    print("Testing filenams:")
    for filename in file_list:
        print(f"  {rename_func(filename, event)}")
    
    # 3. Setup config
    config = {
        "data_acquisition_method": "s3",
        #"raw_data_bucket": BUCKET,
        #"raw_data_prefix": PATH_OLD,
        "cog_data_bucket": BUCKET,
        "cog_data_prefix": f'{DIR_NEW_BASE}/{target_dir}',
        "local_output_dir": f"output/{event}",
        "transformation": {}
    }
    return_bucket_info(config)
    
    # 4. Process files
    print("\n" + "="*50)
    print("🌊 Processing Files (Chunked)")
    print("="*50)
    
    def chunked_converter(local_download_path, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            local_download_path, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, COG_PROFILE,
            local_output_dir, chunk_config=CHUNK_CONFIG
        )

    results = process_file_batch(
        file_list=file_list,
        s3_client=s3_client,
        config=config,
        filename_creator_func=rename_func,
        processing_func=chunked_converter,
        event_name=event,
        save_metadata=True,
        save_csv=True,
        verbose=True,
        BUCKET=BUCKET
    )
    
    print_batch_summary(results)
    return results

In [14]:
keys = [x for x in glob.glob(f"{local_path}*") if x.endswith(".tif")]

reg_keys = make_regex_dict(keys, [r".*_colorInfrared_.*.tif", r".*_trueColor_.*.tif", r".*_naturalColor_.*.tif"], ["colorInfrared", "trueColor", "naturalColor"])

In [15]:
print(reg_keys)
for k, v in reg_keys.items():
    for filename in v:
        print(create_cog_filename(filename, EVENT_NAME))

{'colorInfrared': ['/home/jovyan/files_to_manually_convert/LC09_colorInfrared_20250219_merged.tif', '/home/jovyan/files_to_manually_convert/LC09_colorInfrared_20250203_merged.tif', '/home/jovyan/files_to_manually_convert/LC09_colorInfrared_20250217_merged.tif'], 'trueColor': ['/home/jovyan/files_to_manually_convert/LC09_trueColor_20250203_merged.tif', '/home/jovyan/files_to_manually_convert/LC09_trueColor_20250217_merged.tif', '/home/jovyan/files_to_manually_convert/LC09_trueColor_20250219_merged.tif'], 'naturalColor': ['/home/jovyan/files_to_manually_convert/LC09_naturalColor_20250203_merged.tif', '/home/jovyan/files_to_manually_convert/LC09_naturalColor_20250219_merged.tif']}
202502_Flood_OhioValley_LC09_colorInfrared_merged_2025-02-19_day.tif
202502_Flood_OhioValley_LC09_colorInfrared_merged_2025-02-03_day.tif
202502_Flood_OhioValley_LC09_colorInfrared_merged_2025-02-17_day.tif
202502_Flood_OhioValley_LC09_trueColor_merged_2025-02-03_day.tif
202502_Flood_OhioValley_LC09_trueColor_me

In [16]:
for k, v in reg_keys.items():
    results = simple_process_files(file_list = v, rename_func = create_cog_filename, target_dir = f"Landsat/{k}", event = EVENT_NAME)


Testing filenams:
  202502_Flood_OhioValley_LC09_colorInfrared_merged_2025-02-19_day.tif
  202502_Flood_OhioValley_LC09_colorInfrared_merged_2025-02-03_day.tif
  202502_Flood_OhioValley_LC09_colorInfrared_merged_2025-02-17_day.tif
Configuration loaded:
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/Landsat/colorInfrared

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202502_Flood_OhioValley

[1/3] Processing: /home/jovyan/files_to_manually_convert/LC09_colorInfrared_20250203_merged.tif
   Output filename: 202502_Flood_OhioValley_LC09_colorInfrared_merged_2025-02-03_day.tif
   [CACHE HIT] Using local file: /home/jovyan/files_to_manually_convert/LC09_colorInfrared_20250203_merged.tif
   [MEMORY] Initial: 304.0 MB
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Optimal chunk size: 1024x1024
   Estimated memory per chunk: 3.00 MB
   [NODATA] RGB file detected with nodata=0, treating as regular RGB without nodata
   [CHUNKS] P

Band 1:  64%|██████▍   | 85/132 [00:03<00:02, 21.94chunks/s]


   [MEMORY] High usage: 627.8 MB, forcing cleanup...


Band 1:  73%|███████▎  | 96/132 [00:03<00:01, 22.67chunks/s]


   [MEMORY] High usage: 666.0 MB, forcing cleanup...


...

   [VERIFY] Checking reprojected data...
   [VERIFY] RGB file without nodata - all pixel values are valid
   [VERIFY] Band 1: min=1, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 2: min=1, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 3: min=1, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmp8i9mw97t_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmph2d15nax.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Landsat/naturalColor/202502_Flood_OhioValley_LC09_naturalColor_merged_2025-02-19_day.tif
   [MEMORY] Final: 1085.5 MB (Change: +18.8 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202502_Flood_OhioValley_LC09_naturalColor_merged_2025-02-19_day.tif

✅ Batch processing complete: 2 files processed
📁 COGs saved locally to: output/202502_Flood_OhioValley

📊 BATCH PROCESSING SUMMARY
Total files processed: 2
Successful: 2
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-22T21:05:08.381694
